# Lab 05: Conversation Memory -- Solution

**Goal:** Understand why LLMs are stateless and how to add memory by maintaining a message history.

**What you'll learn:**
- Why LLMs forget everything between calls
- How to maintain a message list for conversation memory
- How message history enables multi-turn conversations
- The role of SystemMessage in setting assistant persona

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatGroq(model="llama-3.3-70b-versatile")

## Step 1: LLMs are stateless — they forget!

Each LLM call is independent. The model has NO memory of previous calls. Watch what happens:

In [ ]:
response1 = llm.invoke([HumanMessage(content="My name is Priya and I work in the Bangalore office.")])
print(f"Turn 1: {response1.content[:150]}")

response2 = llm.invoke([HumanMessage(content="What is my name and where do I work?")])
print(f"Turn 2: {response2.content[:150]}")
print("\n\u2192 The LLM has NO memory of the previous message!")

## Step 2: Add memory with a message list

The fix: send the FULL conversation history with each call. We maintain a Python list of messages.

In [ ]:
conversation = [
    SystemMessage(content="You are a helpful UniGPS HR assistant. Be concise."),
    HumanMessage(content="My name is Priya and I work in the Bangalore office."),
]

response1 = llm.invoke(conversation)
print(f"Turn 1: {response1.content[:150]}")

conversation.append(AIMessage(content=response1.content))
conversation.append(HumanMessage(content="What is my name and where do I work?"))
response2 = llm.invoke(conversation)
print(f"Turn 2: {response2.content[:150]}")
print("\n\u2192 With the full message list, the LLM remembers!")

## Step 3: Multi-turn conversation

Let's have a longer conversation and see memory in action.

In [ ]:
history = [
    SystemMessage(content="You are a UniGPS IT support assistant. Be helpful and concise."),
]

questions = [
    "I need a new monitor for my desk.",
    "What's the budget limit for it?",
    "How do I submit the request?",
    "Thanks! One more thing \u2014 can I also get a keyboard?",
]

for q in questions:
    history.append(HumanMessage(content=q))
    response = llm.invoke(history)
    history.append(AIMessage(content=response.content))
    print(f"User: {q}")
    print(f"AI:   {response.content[:150]}")
    print()

## Step 4: Inspect the history

Let's see what the message list looks like.

In [ ]:
print(f"Total messages in history: {len(history)}")
for i, msg in enumerate(history):
    role = msg.type.upper()
    preview = msg.content[:50].replace('\n', ' ')
    print(f"  [{i}] {role:8s}: {preview}...")

## Step 5: The problem with unlimited history

In [ ]:
total_chars = sum(len(m.content) for m in history)
print(f"Total characters in history: {total_chars}")
print(f"Total messages: {len(history)}")

## TODO 1: Memory Correction Test (SOLUTION)

In [ ]:
correction_history = [
    SystemMessage(content="You are a UniGPS HR assistant. Be concise and precise."),
    HumanMessage(content="I'm Priya Sharma from the Mumbai office."),
]
r1 = llm.invoke(correction_history)
correction_history.append(AIMessage(content=r1.content))
print(f"Turn 1 — stated Mumbai: {r1.content[:120]}")

# Now CORRECT the info
correction_history.append(HumanMessage(content="Actually, I transferred last month. I'm now in the Bangalore office."))
r2 = llm.invoke(correction_history)
correction_history.append(AIMessage(content=r2.content))
print(f"Turn 2 — corrected to Bangalore: {r2.content[:120]}")

# Test: which office does the LLM remember?
correction_history.append(HumanMessage(content="Which office am I in?"))
r3 = llm.invoke(correction_history)
correction_history.append(AIMessage(content=r3.content))
print(f"Turn 3 — verification: {r3.content[:120]}")
print(f"\nTotal messages in history: {len(correction_history)}")
print("→ LLMs prioritize the most recent information in the context window!")

## TODO 2: Chat Loop (demo)

Non-interactive demo using a fixed list of questions.

In [ ]:
chat_history = [
    SystemMessage(content="You are a helpful UniGPS assistant. Be concise."),
]
demo_questions = ["Hi, I'm new here!", "What's the WFH policy?", "Thanks!"]
for q in demo_questions:
    chat_history.append(HumanMessage(content=q))
    response = llm.invoke(chat_history)
    chat_history.append(AIMessage(content=response.content))
    print(f"You: {q}")
    print(f"AI:  {response.content[:150]}")
    print()

## Summary

In [ ]:
print("Lab 05 complete! Key takeaways:")
print("- LLMs are stateless \u2014 they don't remember between calls")
print("- A message list (conversation history) adds memory")
print("- Each call sends the FULL history to the LLM")
print("- SystemMessage sets the assistant's persona and rules")
print("- History grows with every turn \u2014 needs management (Lab 06)")